# 03 - Phân Tích Thống Kê Khám Phá
## Mental Health Survey Dataset

**Mô tả:** EDA cho tập dữ liệu khảo sát sức khỏe tâm thần với 140,700 bản ghi và 19 thuộc tính.
**Mục tiêu:** Phân tích phân phối, tương quan và giá trị thiếu trước khi tiền xử lý.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
from scipy.stats import normaltest, chi2
import warnings
warnings.filterwarnings('ignore')
import os

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
os.makedirs('../data/processed', exist_ok=True)

# Load dataset once so downstream cells can use df safely
df = pd.read_csv('../data/raw/mental_health_train.csv')
print(f'Data shape: {df.shape}')

## 1. Kiểm Tra Phân Phối: D'Agostino-Pearson Test

**D'Agostino-Pearson test** kết hợp skewness và kurtosis để kiểm tra chuẩn.
- H₀: dữ liệu có phân phối chuẩn
- Dùng D'Agostino-Pearson khi **n > 5,000** (thay vì Shapiro-Wilk chỉ phù hợp n <= 5,000)
- Kết quả phân loại được dùng để **chọn phương pháp chuẩn hóa phù hợp**


### D'AGOSTINO-PEARSON NORMALITY TEST

In [ ]:
from scipy.stats import normaltest

numerical_cols = [
    c
    for c in df.columns
    if df[c].dtype in [np.float64, np.int64] and c not in ["id", "Depression"]
]

print(f"{'Thuộc tính':<35} {'n':>8} {'Statistic':>12} {'p-value':>12} {'Kết quả':>12}")

normality_results = {}
for col in numerical_cols:
    data = df[col].dropna()
    n = len(data)
    if n < 20:
        continue
    stat, p = normaltest(data)
    is_normal = p > 0.05
    normality_results[col] = {
        "n": int(n),
        "statistic": round(float(stat), 4),
        "p_value": round(float(p), 8),
        "is_normal": is_normal,
        "distribution": "Chuẩn" if is_normal else "Không chuẩn",
        "skewness": round(float(data.skew()), 4),
        "kurtosis": round(float(data.kurtosis()), 4),
    }
    flag = "CHUẨN" if is_normal else "KHÔNG CHUẨN"
    print(f"{col:<35} {n:>8,} {stat:>12.4f} {p:>12.2e} {flag:>12}")



### Phân Tích Kết Quả: Kiểm Tra Phân Phối D'Agostino-Pearson

**Kết quả thực tế:**

| Thuộc tính | n | p-value | Skewness | Kurtosis | Kết luận |
|---|---|---|---|---|---|
| Age | 140,700 | ~0.0 | -0.218 | -1.149 | Không chuẩn |
| Academic Pressure | 27,897 | ~0.0 | -0.134 | -1.164 | Không chuẩn |
| Work Pressure | 112,782 | ~0.0 | 0.019 | -1.295 | Không chuẩn |
| CGPA | 27,898 | ~0.0 | -0.074 | -1.229 | Không chuẩn |
| Study Satisfaction | 27,897 | ~0.0 | 0.012 | -1.226 | Không chuẩn |
| Job Satisfaction | 112,790 | ~0.0 | 0.054 | -1.305 | Không chuẩn |
| Work/Study Hours | 140,700 | ~0.0 | -0.128 | -1.283 | Không chuẩn |
| Financial Stress | 140,696 | ~0.0 | 0.036 | -1.314 | Không chuẩn |

**Nhận xét:**
- **Tất cả 8 thuộc tính số đều KHÔNG phân phối chuẩn** (p~0 với n rất lớn).
- **Kurtosis ~ -1.3 đến -1.1** trên tất cả các cột -> phân phối có đỉnh phẳng (platykurtic), gần với phân phối đều (uniform). Điều này hợp lý vì các thang đo ratings (1-5) có xu hướng phân bố đều.
- **Skewness gần 0** -> không có lệch rõ ràng, ngoại trừ Age (skew=-0.218, lệch nhẹ về trái).
- **Lý do không chuẩn:** Với n=140,700, D'Agostino-Pearson test rất nhạy cảm — ngay cả lệch rất nhỏ cũng bị phát hiện là không chuẩn. Kurtosis âm (-1.2) cho thấy phân phối "rộng và phẳng" hơn phân phối chuẩn.

**Khuyến nghị chuẩn hóa:**
-> Vì dữ liệu không chuẩn và có kurtosis âm, nên dùng **Robust Scaling** hoặc **Quantile Transform** thay vì Z-score (chỉ tốt với phân phối chuẩn).


In [ ]:
cols_to_plot = [c for c in numerical_cols if normality_results.get(c)]
n_cols = len(cols_to_plot)
n_rows = (n_cols + 2) // 3

fig, axes = plt.subplots(n_rows, 3, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    data = df[col].dropna()
    res = normality_results[col]
    axes[i].hist(data, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
    axes[i].set_title(
        f"{col}\nn={res['n']:,} | p={res['p_value']:.2e} | "
        f"{'Chuẩn' if res['is_normal'] else 'Không chuẩn'}\n"
        f"skew={res['skewness']:.2f}, kurt={res['kurtosis']:.2f}",
        fontsize=9,
    )
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel("Tần suất", fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Phân Phối Các Thuộc Tính Số — Mental Health Dataset", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../data/processed/distribution_plots.png", dpi=100, bbox_inches="tight")
plt.show()

## 2. Phân Tích Tương Quan Đa Biến

- **Pearson**: đo tương quan tuyến tính, phù hợp khi dữ liệu phân phối chuẩn
- **Spearman**: đo tương quan đơn điệu (rank-based), phù hợp khi dữ liệu không chuẩn
- Ngưỡng đa cộng tuyến mạnh: **|r| > 0.9** -> cần xử lý


In [ ]:
corr_cols = [c for c in numerical_cols if df[c].notna().sum() > len(df) * 0.1]

df_corr = df[corr_cols + ["Depression"]].copy()

pearson_corr = df_corr.corr(method="pearson")
spearman_corr = df_corr.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(
    pearson_corr,
    annot=True,
    fmt=".2f",
    ax=axes[0],
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    annot_kws={"size": 8},
)
axes[0].set_title("Pearson Correlation Heatmap", fontsize=13, pad=12)
axes[0].tick_params(axis="x", rotation=45, labelsize=8)
axes[0].tick_params(axis="y", rotation=0, labelsize=8)

sns.heatmap(
    spearman_corr,
    annot=True,
    fmt=".2f",
    ax=axes[1],
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    annot_kws={"size": 8},
)
axes[1].set_title("Spearman Correlation Heatmap", fontsize=13, pad=12)
axes[1].tick_params(axis="x", rotation=45, labelsize=8)
axes[1].tick_params(axis="y", rotation=0, labelsize=8)

plt.suptitle("Heatmap Tương Quan — Mental Health Dataset", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../data/processed/correlation_heatmaps.png", dpi=100, bbox_inches="tight")
plt.show()

### ĐA CỘNG TUYẾN MẠNH (|Pearson r| > 0.9)

In [ ]:
threshold = 0.9
found = False
for i in range(len(pearson_corr.columns)):
    for j in range(i + 1, len(pearson_corr.columns)):
        r = pearson_corr.iloc[i, j]
        if abs(r) > threshold:
            print(
                f"  {pearson_corr.columns[i]} <-> {pearson_corr.columns[j]}: r = {r:.4f}"
            )
            found = True
if not found:
    print("  Không có cặp thuộc tính nào có |r| > 0.9")
    print("  -> Dữ liệu KHÔNG có đa cộng tuyến mạnh")

print("Tương quan với biến mục tiêu (Depression)")
target_corr_p = (
    pearson_corr["Depression"].drop("Depression").abs().sort_values(ascending=False)
)
target_corr_s = (
    spearman_corr["Depression"].drop("Depression").abs().sort_values(ascending=False)
)
print(f"{'Thuộc tính':<35} {'|Pearson|':>10} {'|Spearman|':>12}")
for col in target_corr_p.index:
    p_r = target_corr_p[col]
    s_r = target_corr_s[col] if col in target_corr_s else float("nan")
    print(f"  {col:<33} {p_r:>10.4f} {s_r:>12.4f}")



### Phân Tích Kết Quả: Tương Quan Đa Biến

**Kết quả thực tế — Pearson và Spearman với Depression:**

| Thuộc tính | Pearson r | Spearman ρ | Độ mạnh |
|---|---|---|---|
| **Age** | **-0.565** | **-0.541** | **Trung bình mạnh** |
| Academic Pressure | +0.475 | +0.472 | Trung bình |
| Financial Stress | +0.227 | +0.227 | Yếu |
| Work Pressure | +0.217 | +0.216 | Yếu |
| Work/Study Hours | +0.192 | +0.192 | Yếu |
| Job Satisfaction | -0.169 | -0.170 | Yếu |
| Study Satisfaction | -0.168 | -0.168 | Yếu |
| CGPA | +0.022 | +0.022 | Rất yếu |

**Phát hiện quan trọng — Artifact MAR:**
- Xuất hiện **tương quan hoàn hảo** giữa Academic Pressure <-> Job Satisfaction (r=-1.0) và CGPA <-> Job Satisfaction (r=1.0).
- **Nguyên nhân:** Đây là artifact của cấu trúc dữ liệu MAR. Khi tính Pearson trên toàn dataset với `dropna()`, các bộ quan sát có đồng thời Academic Pressure VÀ Job Satisfaction không null là cực kỳ ít (vì chúng thuộc 2 nhóm khác nhau). Tương quan trên tập nhỏ này trở thành perfect correlation ngẫu nhiên.
- **Xử lý:** Các cặp tương quan hoàn hảo này KHÔNG phản ánh thực tế — cần tính tương quan riêng theo nhóm (Student vs Working Professional).

**Nhận xét thực chất:**
- **Age là predictor mạnh nhất** (r=-0.565): người lớn tuổi ít bị trầm cảm hơn trong dataset này.
- **Academic Pressure** (r=+0.475): áp lực học tập cao -> tăng nguy cơ trầm cảm ở sinh viên.
- Pearson ~ Spearman cho tất cả các cột -> tương quan tuyến tính và đơn điệu nhất quán.
- **Không có đa cộng tuyến thực sự** trong nhóm thuộc tính liên quan đến cùng một nhóm (Student hoặc Professional).


## 3. Phân Tích Giá Trị Thiếu

**Các loại cơ chế thiếu:**
- **MCAR** (Missing Completely At Random): thiếu hoàn toàn ngẫu nhiên, không phụ thuộc vào bất kỳ biến nào
- **MAR** (Missing At Random): thiếu phụ thuộc vào các biến quan sát được (không phải giá trị bị thiếu)
- **MNAR** (Missing Not At Random): thiếu phụ thuộc vào chính giá trị bị thiếu

**Little's MCAR Test**: H₀ = dữ liệu MCAR. p-value nhỏ -> bác bỏ MCAR.


### PHÂN TÍCH GIÁ TRỊ THIẾU — MISSINGNO VISUALIZATION

In [ ]:
missing_stats = pd.DataFrame(
    {
        "Missing Count": df.isnull().sum(),
        "Missing %": (df.isnull().sum() / len(df) * 100).round(2),
    }
).sort_values("Missing %", ascending=False)
missing_stats = missing_stats[missing_stats["Missing Count"] > 0]
print(missing_stats.to_string())

In [ ]:
sample_size = min(10000, len(df))
df_sample = df.sample(sample_size, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plt.sca(axes[0])
msno.matrix(
    df_sample, ax=axes[0], sparkline=False, fontsize=9, color=(0.27, 0.52, 0.71)
)
axes[0].set_title(f"Missing Data Matrix (n={sample_size:,})", fontsize=12)

plt.sca(axes[1])
msno.bar(df_sample, ax=axes[1], fontsize=9, color="steelblue")
axes[1].set_title("Missing Data Bar Chart", fontsize=12)

plt.tight_layout()
plt.savefig("../data/processed/missing_data_analysis.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
msno.heatmap(df_sample, ax=ax, fontsize=9)
ax.set_title("Correlation of Missing Values (missingno heatmap)", fontsize=12)
plt.tight_layout()
plt.savefig(
    "../data/processed/missing_correlation_heatmap.png", dpi=100, bbox_inches="tight"
)
plt.show()

### LITTLE'S MCAR TEST

In [ ]:
def little_mcar_test(data):
    """
    Approximation of Little's MCAR test.
    Uses chi-square statistic comparing observed vs expected covariance patterns.
    """
    numeric_data = data.select_dtypes(include=[np.number]).copy()
    cols_with_missing = [
        c for c in numeric_data.columns if numeric_data[c].isnull().any()
    ]

    if len(cols_with_missing) == 0:
        return None, None, "No missing values in numerical columns"

    missing_indicators = numeric_data[cols_with_missing].isnull().astype(int)

    patterns = missing_indicators.apply(lambda row: tuple(row), axis=1)
    pattern_groups = patterns.groupby(patterns)

    overall_means = numeric_data.mean()

    d2 = 0
    total_df = 0
    n_patterns = 0

    for pattern, idx in pattern_groups.groups.items():
        group = numeric_data.loc[idx]
        observed_cols = [cols_with_missing[i] for i, v in enumerate(pattern) if v == 0]
        if len(observed_cols) == 0 or len(group) < 2:
            continue
        group_means = group[observed_cols].mean()
        pop_means = overall_means[observed_cols]
        n_j = len(group)
        diff = group_means - pop_means
        try:
            cov = group[observed_cols].cov()
            cov_inv = np.linalg.pinv(cov.values)
            d2 += n_j * (diff.values @ cov_inv @ diff.values)
            total_df += len(observed_cols)
            n_patterns += 1
        except Exception:
            pass

    if total_df == 0:
        return None, None, "Could not compute test statistic"

    p_value = 1 - chi2.cdf(d2, df=total_df)
    return d2, p_value, f"chi2={d2:.4f}, df={total_df}, p={p_value:.6f}"


test_cols = [
    "Age",
    "Work Pressure",
    "Job Satisfaction",
    "Work/Study Hours",
    "Financial Stress",
]
df_test = df[test_cols].copy()

chi2_stat, p_val, result_str = little_mcar_test(df_test)
print(f"Test Statistic: {result_str}")

if p_val is not None:
    if p_val > 0.05:
        print(
            f"p-value = {p_val:.6f} > 0.05 -> KHÔNG bác bỏ H₀ -> Dữ liệu có thể là MCAR"
        )
        mcar_conclusion = "MCAR"
    else:
        print(f"p-value = {p_val:.6f} <= 0.05 -> Bác bỏ H₀ -> Dữ liệu KHÔNG phải MCAR")
        mcar_conclusion = "MAR/MNAR"

### PHÂN LOẠI CƠ CHẾ THIẾU DỮ LIỆU (MCAR / MAR / MNAR)

In [ ]:
print("\nKiểm tra Work Pressure:")
wp_prof = (
    df[df["Working Professional or Student"] == "Working Professional"]["Work Pressure"]
    .isnull()
    .sum()
)
wp_student = (
    df[df["Working Professional or Student"] == "Student"]["Work Pressure"]
    .isnull()
    .sum()
)
wp_total_missing = df["Work Pressure"].isnull().sum()
print(f"  Missing trong Working Professional: {wp_prof}")
print(f"  Missing trong Student: {wp_student}")
print(
    f"  -> Hầu hết thiếu ở nhóm Student -> MAR (phụ thuộc vào 'Working Professional or Student')"
)

print("\nKiểm tra Academic Pressure:")
ap_prof = (
    df[df["Working Professional or Student"] == "Working Professional"][
        "Academic Pressure"
    ]
    .isnull()
    .sum()
)
ap_student = (
    df[df["Working Professional or Student"] == "Student"]["Academic Pressure"]
    .isnull()
    .sum()
)
print(f"  Missing trong Working Professional: {ap_prof}")
print(f"  Missing trong Student: {ap_student}")
print(f"  -> Hầu hết thiếu ở nhóm Working Professional -> MAR")

print("\nKiểm tra Profession:")
pf_student = (
    df[df["Working Professional or Student"] == "Student"]["Profession"].isnull().sum()
)
pf_prof = (
    df[df["Working Professional or Student"] == "Working Professional"]["Profession"]
    .isnull()
    .sum()
)
print(f"  Missing trong Student: {pf_student}")
print(f"  Missing trong Working Professional: {pf_prof}")
print(f"  -> Thiếu ở nhóm Student (sinh viên chưa có nghề) -> MAR")

print("\nKiểm tra Dietary Habits / Financial Stress:")
for col in ["Dietary Habits", "Financial Stress"]:
    miss = df[col].isnull().sum()
    pct = miss / len(df) * 100
    print(
        f"  {col}: {miss} missing ({pct:.3f}%) -> Rất ít, có thể là MCAR hoặc nhập liệu thiếu sót"
    )

print("BẢNG TỔNG HỢP CƠ CHẾ THIẾU DỮ LIỆU")
missing_mechanism = {
    "Academic Pressure": (
        "80.17%",
        "MAR",
        "Chỉ sinh viên có Academic Pressure; Working Professional -> missing",
    ),
    "CGPA": ("80.17%", "MAR", "Chỉ sinh viên có CGPA; Working Professional -> missing"),
    "Study Satisfaction": (
        "80.17%",
        "MAR",
        "Chỉ sinh viên có Study Satisfaction; Working Professional -> missing",
    ),
    "Work Pressure": (
        "19.84%",
        "MAR",
        "Chỉ Working Professional có Work Pressure; Student -> missing",
    ),
    "Job Satisfaction": (
        "19.84%",
        "MAR",
        "Chỉ Working Professional có Job Satisfaction; Student -> missing",
    ),
    "Profession": ("26.03%", "MAR", "Sinh viên chưa có nghề nghiệp cụ thể -> missing"),
    "Dietary Habits": ("0.003%", "MCAR", "Rất ít, ngẫu nhiên không có hệ thống"),
    "Financial Stress": ("0.003%", "MCAR", "Rất ít, ngẫu nhiên không có hệ thống"),
    "Degree": ("0.001%", "MCAR", "Rất ít, ngẫu nhiên không có hệ thống"),
}

print(f"{'Thuộc tính':<25} {'% Thiếu':>10} {'Cơ chế':>8}  Giải thích")
for col, (pct, mech, reason) in missing_mechanism.items():
    print(f"  {col:<23} {pct:>10} {mech:>8}  {reason}")


> **KẾT LUẬN:** Phần lớn giá trị thiếu là MAR (phụ thuộc vào nhóm nghề/sinh viên). Không nên impute trực tiếp toàn bộ dataset — nên xử lý theo nhóm (Working Professional vs Student) hoặc dùng group-wise imputation.

## 4. Tổng Kết EDA

Dựa trên EDA:
1. **Phân phối**: Hầu hết thuộc tính số không phân phối chuẩn (Non-Normal) -> dùng Robust Scaling / Quantile Transform
2. **Tương quan**: Không có đa cộng tuyến mạnh (|r| > 0.9), tuy nhiên Academic Pressure <-> Study Satisfaction và Work Pressure <-> Job Satisfaction có tương quan đáng kể
3. **Giá trị thiếu**: Chủ yếu là **MAR** (phụ thuộc nhóm Professional/Student), không phải MCAR
4. **Mất cân bằng**: 18.2% positive class (Depression=1) — cần kỹ thuật xử lý imbalance
5. **Hướng xử lý**: Impute theo nhóm, sử dụng Robust/Quantile normalization, xử lý imbalance với SMOTE/ADASYN


### BẢNG TÓM TẮT KẾT QUẢ EDA

In [ ]:
print("\n1. PHÂN PHỐI:")
for col, res in normality_results.items():
    print(
        f"   {col:<35}: {res['distribution']} (skew={res['skewness']:.2f}, p={res['p_value']:.2e})"
    )

print("\n2. GIÁ TRỊ THIẾU:")
print("   MAR: Academic Pressure, CGPA, Study Satisfaction, Work Pressure,")
print("          Job Satisfaction, Profession (do chia nhóm Student/Professional)")
print("   MCAR: Dietary Habits, Financial Stress, Degree (rất ít, ngẫu nhiên)")

print("\n3. BIẾN MỤC TIÊU:")
dep = df["Depression"].value_counts()
print(
    f"   Depression=0: {dep[0]:,} ({dep[0]/len(df)*100:.1f}%) | Depression=1: {dep[1]:,} ({dep[1]/len(df)*100:.1f}%)"
)
print(f"   -> Mất cân bằng lớp rõ rệt (tỉ lệ 4.5:1)")